# 25. Bag of Words

## Overview
The Bag of Words (BoW) model represents a document by the counts of words it contains. It ignores word order, making it simple and effective as a baseline for many text classification tasks.

This notebook builds a BoW representation, inspects its vocabulary, and demonstrates common configuration choices.

## 1. Import and prepare documents

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "The product is excellent and easy to use",
    "The product is affordable and useful",
    "The service is slow and disappointing",
    "Excellent service and excellent support",
]

## 2. Build the Bag of Words matrix

In [2]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

bow_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(1, len(documents) + 1)],
)

display(bow_df)

,affordable,and,disappointing,easy,excellent,is,product,service,slow,support,the,to,use,useful
doc_1,0,1,0,1,1,1,1,0,0,0,1,1,1,0
doc_2,1,1,0,0,0,1,1,0,0,0,1,0,0,1
doc_3,0,1,1,0,0,1,0,1,1,0,1,0,0,0
doc_4,0,1,0,0,2,0,0,1,0,1,0,0,0,0


## 3. Inspect the vocabulary

The vectorizer assigns a feature index to each term.

In [3]:
vocabulary = vectorizer.vocabulary_
for term, index in sorted(vocabulary.items(), key=lambda item: item[1]):
    print(f"{index:>2}: {term}")

 0: affordable
 1: and
 2: disappointing
 3: easy
 4: excellent
 5: is
 6: product
 7: service
 8: slow
 9: support
10: the
11: to
12: use
13: useful


## 4. Configure token filtering

`min_df` removes terms that occur in too few documents; `max_df` removes terms that occur in a large proportion of documents. These settings should be selected using training data.

In [4]:
filtered_vectorizer = CountVectorizer(
    lowercase=True,
    min_df=1,
    max_df=1.0,
    ngram_range=(1, 1),
)

X_filtered = filtered_vectorizer.fit_transform(documents)
print("Vocabulary size:", len(filtered_vectorizer.get_feature_names_out()))
print(filtered_vectorizer.get_feature_names_out())

Vocabulary size: 14
['affordable' 'and' 'disappointing' 'easy' 'excellent' 'is' 'product'
 'service' 'slow' 'support' 'the' 'to' 'use' 'useful']


## 5. Include bigrams

Unigrams are individual tokens. Bigrams are adjacent two-token sequences and can preserve limited phrase information.

In [5]:
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_ngrams = ngram_vectorizer.fit_transform(documents)

ngram_features = ngram_vectorizer.get_feature_names_out()
print("Total unigram + bigram features:", len(ngram_features))
print("Sample features:", ngram_features[:30])

Total unigram + bigram features: 33
Sample features: ['affordable' 'affordable and' 'and' 'and disappointing' 'and easy'
 'and excellent' 'and useful' 'disappointing' 'easy' 'easy to' 'excellent'
 'excellent and' 'excellent service' 'excellent support' 'is'
 'is affordable' 'is excellent' 'is slow' 'product' 'product is' 'service'
 'service and' 'service is' 'slow' 'slow and' 'support' 'the'
 'the product' 'the service' 'to']


## 6. Understand the limitations

BoW discards word order and semantic relationships. For example, `not good` and `good` share the feature `good`; using bigrams may help represent some local context but does not solve the full problem.

In [6]:
example_docs = ["the movie was good", "the movie was not good"]
example_vectorizer = CountVectorizer(ngram_range=(1, 2))
example_X = example_vectorizer.fit_transform(example_docs)

display(pd.DataFrame(
    example_X.toarray(),
    columns=example_vectorizer.get_feature_names_out(),
    index=["positive wording", "negated wording"],
))

,good,movie,movie was,not,not good,the,the movie,was,was good,was not
positive wording,1,1,1,0,0,1,1,1,1,0
negated wording,1,1,1,1,1,1,1,1,0,1


## Key takeaways

- BoW creates one feature per vocabulary term.
- Counts are easy to inspect and form a strong baseline.
- N-grams can capture short phrases.
- Vocabulary size and lack of deeper semantic context are important limitations.